# Scikit-learn Study Guide

Machine Learning in Python — from data loading to model deployment.

**Topics:** Setup & Data Loading, Linear & Logistic Regression, Decision Trees & Random Forest, SVM, KNN & Naive Bayes, Clustering, Metrics, Pipelines, Hyperparameter Tuning, PCA & t-SNE

## Setup & Data Loading

Import scikit-learn, load built-in datasets, split data into train/test sets.

### Loading Datasets & Train/Test Split

In [ ]:
from sklearn.datasets import load_iris, load_boston, make_classification
from sklearn.model_selection import train_test_split
import pandas as pd

# Load Iris dataset
iris = load_iris()
X, y = iris.data, iris.target
print('Features:', iris.feature_names)
print('Classes:', iris.target_names)
print('Shape:', X.shape)

# Split 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

### Synthetic Dataset with make_classification

In [ ]:
from sklearn.datasets import make_classification, make_regression
import numpy as np

# Synthetic classification data
X, y = make_classification(
    n_samples=1000, n_features=10,
    n_informative=5, n_redundant=2,
    random_state=42
)
print('X shape:', X.shape, '| Classes:', np.unique(y))

# Synthetic regression data
X_r, y_r = make_regression(
    n_samples=500, n_features=5, noise=0.1, random_state=42
)
print('Regression X:', X_r.shape, '| y range:', y_r.min().round(1), '-', y_r.max().round(1))

### Real-World Use Case

**Scenario:** Healthcare: Load patient vitals dataset, split into train/test while preserving class balance (stratify) for disease prediction.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Simulate patient vitals
import numpy as np
np.random.seed(42)
df = pd.DataFrame({
    'age': np.random.randint(20, 80, 200),
    'bp': np.random.randint(60, 140, 200),
    'cholesterol': np.random.randint(150, 300, 200),
    'glucose': np.random.randint(70, 200, 200),
    'disease': np.random.choice([0, 1], 200, p=[0.7, 0.3])
})
X = df.drop('disease', axis=1)
y = df['disease']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f'Train positives: {y_train.mean():.1%} | Test positives: {y_test.mean():.1%}')

## Linear & Logistic Regression

LinearRegression for continuous targets; LogisticRegression for binary/multiclass classification.

### Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

X, y = make_regression(n_samples=200, n_features=3, noise=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print('Coefficients:', model.coef_.round(2))
print('Intercept:', round(model.intercept_, 2))
print('R2 Score:', round(r2_score(y_test, y_pred), 4))
print('RMSE:', round(np.sqrt(mean_squared_error(y_test, y_pred)), 2))

### Logistic Regression (Classification)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target
)

lr = LogisticRegression(max_iter=200)
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

print('Accuracy:', accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=iris.target_names))

### Real-World Use Case

**Scenario:** Finance: Predict house prices (Linear Regression) and loan default probability (Logistic Regression).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
import numpy as np

# Simulate loan applicant data
np.random.seed(42)
n = 500
income = np.random.normal(50000, 15000, n)
debt = np.random.normal(20000, 8000, n)
credit_score = np.random.randint(300, 850, n)
X = np.column_stack([income, debt, credit_score])
# Default if debt/income > 0.6 or credit < 550
y = ((debt / income > 0.6) | (credit_score < 550)).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LogisticRegression()
model.fit(X_train, y_train)
print('Loan Default Prediction:')
print(classification_report(y_test, model.predict(X_test)))

## Decision Trees & Random Forest

Tree-based models: interpretable Decision Trees and powerful ensemble Random Forests.

### Decision Tree Classifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)

dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X_train, y_train)
print('Accuracy:', accuracy_score(y_test, dt.predict(X_test)))
print('Feature importances:')
for name, imp in zip(iris.feature_names, dt.feature_importances_):
    print(f'  {name}: {imp:.3f}')
print(export_text(dt, feature_names=iris.feature_names))

### Random Forest & Feature Importance

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np

X, y = make_classification(n_samples=1000, n_features=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
print('RF Accuracy:', accuracy_score(y_test, rf.predict(X_test)))

# Top 5 features
idx = np.argsort(rf.feature_importances_)[::-1][:5]
for i in idx:
    print(f'  Feature {i}: {rf.feature_importances_[i]:.4f}')

### Real-World Use Case

**Scenario:** Retail: Predict customer churn using Random Forest — who is likely to stop buying?

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import numpy as np, pandas as pd

np.random.seed(42)
n = 1000
df = pd.DataFrame({
    'recency_days': np.random.randint(1, 365, n),
    'frequency': np.random.randint(1, 50, n),
    'monetary': np.random.exponential(200, n),
    'tenure_months': np.random.randint(1, 60, n),
    'support_calls': np.random.poisson(2, n)
})
# Churn if high recency, low frequency
df['churn'] = ((df['recency_days'] > 200) & (df['frequency'] < 5)).astype(int)

X = df.drop('churn', axis=1)
y = df['churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
print('Customer Churn Prediction:')
print(classification_report(y_test, rf.predict(X_test)))

## Support Vector Machines

SVM finds the maximum-margin hyperplane. Works for classification (SVC) and regression (SVR).

### SVC with Kernel Trick

In [ ]:
from sklearn.svm import SVC
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

X, y = make_classification(n_samples=500, n_features=5, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# SVM needs scaled features
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

for kernel in ['linear', 'rbf', 'poly']:
    svm = SVC(kernel=kernel, C=1.0)
    svm.fit(X_train_s, y_train)
    acc = accuracy_score(y_test, svm.predict(X_test_s))
    print(f'{kernel:8s} kernel accuracy: {acc:.4f}')

### SVR for Regression

In [ ]:
from sklearn.svm import SVR
from sklearn.datasets import make_regression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

X, y = make_regression(n_samples=300, n_features=3, noise=15, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

for kernel in ['linear', 'rbf']:
    svr = SVR(kernel=kernel, C=10)
    svr.fit(X_train_s, y_train)
    r2 = r2_score(y_test, svr.predict(X_test_s))
    print(f'SVR ({kernel}) R2: {r2:.4f}')

### Real-World Use Case

**Scenario:** NLP: SVM for text sentiment classification — positive vs negative product reviews.

In [ ]:
from sklearn.svm import SVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

# Simulated reviews
reviews = [
    'great product love it', 'terrible waste of money',
    'amazing quality highly recommend', 'broken arrived damaged',
    'best purchase ever', 'awful customer service never again',
    'works perfectly fast shipping', 'poor quality disappointed',
]
labels = [1, 0, 1, 0, 1, 0, 1, 0]  # 1=positive, 0=negative

pipe = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('svm', SVC(kernel='linear', C=1.0))
])
pipe.fit(reviews[:6], labels[:6])
preds = pipe.predict(reviews[6:])
print('Predictions:', ['Positive' if p else 'Negative' for p in preds])
print('True labels:', ['Positive' if l else 'Negative' for l in labels[6:]])

## K-Nearest Neighbors & Naive Bayes

KNN classifies by majority vote of k neighbors. Naive Bayes uses Bayes' theorem with feature independence assumption.

### K-Nearest Neighbors

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
import numpy as np

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Find best k
for k in [1, 3, 5, 7, 11]:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_train_s, y_train, cv=5)
    print(f'k={k:2d}: CV accuracy = {scores.mean():.4f} (+/- {scores.std():.4f})')

### Naive Bayes for Text Classification

In [ ]:
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Email spam detection
emails = [
    'win money now free prize', 'meeting tomorrow at 10am',
    'congratulations you won cash', 'project update attached',
    'free viagra cheap pills', 'please review the report',
    'claim your reward today', 'lunch at noon works for me',
]
labels = [1, 0, 1, 0, 1, 0, 1, 0]  # 1=spam

vec = CountVectorizer()
X = vec.fit_transform(emails)
nb = MultinomialNB()
nb.fit(X, labels)
test = vec.transform(['free money win now', 'schedule a meeting'])
preds = nb.predict(test)
print('Predictions:', ['SPAM' if p else 'HAM' for p in preds])

### Real-World Use Case

**Scenario:** E-commerce: Recommend similar products using KNN — find the 5 most similar items based on features.

In [ ]:
from sklearn.neighbors import NearestNeighbors
import numpy as np, pandas as pd

# Product feature vectors (price, rating, sales, weight)
np.random.seed(42)
products = pd.DataFrame({
    'name': [f'Product_{i}' for i in range(20)],
    'price': np.random.uniform(10, 200, 20),
    'rating': np.random.uniform(1, 5, 20),
    'sales': np.random.randint(100, 10000, 20),
    'weight_kg': np.random.uniform(0.1, 5, 20)
})

from sklearn.preprocessing import StandardScaler
X = StandardScaler().fit_transform(products[['price','rating','sales','weight_kg']])

nn = NearestNeighbors(n_neighbors=4, metric='euclidean')
nn.fit(X)
# Find similar products to Product_0
distances, indices = nn.kneighbors([X[0]])
print('Products similar to', products.iloc[0]['name'])
for i, d in zip(indices[0][1:], distances[0][1:]):
    print(f'  {products.iloc[i]["name"]} (dist={d:.2f})')

## Clustering: KMeans & DBSCAN

Unsupervised learning: group similar data points without labels.

### KMeans Clustering

In [ ]:
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score
import numpy as np

X, _ = make_blobs(n_samples=300, centers=4, random_state=42)

# Elbow method to find optimal k
inertias = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertias.append(km.inertia_)
    sil = silhouette_score(X, km.labels_)
    print(f'k={k}: inertia={km.inertia_:.1f}, silhouette={sil:.3f}')

# Best k=4
best = KMeans(n_clusters=4, random_state=42, n_init=10)
best.fit(X)
print('Cluster sizes:', {i: (best.labels_==i).sum() for i in range(4)})

### DBSCAN for Density-Based Clustering

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler
import numpy as np

# make_moons: non-convex shapes KMeans can't handle
X, _ = make_moons(n_samples=200, noise=0.1, random_state=42)
X = StandardScaler().fit_transform(X)

db = DBSCAN(eps=0.3, min_samples=5)
db.fit(X)

labels = db.labels_
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = (labels == -1).sum()
print(f'Clusters found: {n_clusters}')
print(f'Noise points: {n_noise}')
print(f'Cluster sizes: {[(labels==i).sum() for i in range(n_clusters)]}')

### Real-World Use Case

**Scenario:** Marketing: Segment customers into groups (KMeans) based on RFM (Recency, Frequency, Monetary) for targeted campaigns.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import numpy as np, pandas as pd

np.random.seed(42)
n = 500
df = pd.DataFrame({
    'recency': np.random.randint(1, 365, n),
    'frequency': np.random.randint(1, 100, n),
    'monetary': np.random.exponential(300, n)
})

X = StandardScaler().fit_transform(df)
km = KMeans(n_clusters=4, random_state=42, n_init=10)
df['segment'] = km.fit_predict(X)

segment_names = {0: 'Champions', 1: 'At Risk', 2: 'New Customers', 3: 'Lost'}
summary = df.groupby('segment').agg({'recency':'mean','frequency':'mean','monetary':'mean','segment':'count'})
summary.columns = ['Avg Recency', 'Avg Frequency', 'Avg Monetary', 'Count']
print('Customer Segments:')
print(summary.round(1))

## Model Evaluation & Metrics

Measure model performance: confusion matrix, ROC-AUC, precision-recall, cross-validation.

### Classification Metrics

In [ ]:
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve, accuracy_score
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=500, n_features=8, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestClassifier(n_estimators=50, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print('Accuracy:', accuracy_score(y_test, y_pred))
print('ROC-AUC:', roc_auc_score(y_test, y_prob).round(4))
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

### Cross-Validation & Regression Metrics

In [ ]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.datasets import make_regression
import numpy as np

X, y = make_regression(n_samples=300, n_features=5, noise=20, random_state=42)

rf = RandomForestRegressor(n_estimators=50, random_state=42)

# 5-fold cross-validation
cv = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(rf, X, y, cv=cv, scoring='r2')
print(f'CV R2: {scores.mean():.4f} +/- {scores.std():.4f}')

# Also report MAE, RMSE on test set
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
rf.fit(X_train, y_train)
yp = rf.predict(X_test)
print(f'MAE: {mean_absolute_error(y_test, yp):.2f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, yp)):.2f}')
print(f'R2: {r2_score(y_test, yp):.4f}')

### Real-World Use Case

**Scenario:** Medical: Evaluate a cancer detection model — minimize false negatives (missed cancers) using recall + AUC.

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, recall_score
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification

# Simulate imbalanced cancer screening data (10% positive)
X, y = make_classification(
    n_samples=1000, weights=[0.9, 0.1],
    n_features=8, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

model = GradientBoostingClassifier(random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print('Cancer Detection Model Evaluation:')
print(f'Recall (sensitivity): {recall_score(y_test, y_pred):.4f}')
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}')
print(classification_report(y_test, y_pred, target_names=['Healthy','Cancer']))

## Pipelines & Preprocessing

Chain preprocessing + model into a single Pipeline. Prevent data leakage and simplify deployment.

### Building a Pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
import numpy as np

X, y = make_classification(n_samples=500, n_features=6, random_state=42)
# Inject missing values
X_missing = X.copy()
X_missing[np.random.choice(500, 50), np.random.choice(6, 50)] = np.nan

X_train, X_test, y_train, y_test = train_test_split(X_missing, y, test_size=0.2, random_state=42)

pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(n_estimators=50, random_state=42))
])
pipe.fit(X_train, y_train)
print('Pipeline accuracy:', pipe.score(X_test, y_test).round(4))

### ColumnTransformer for Mixed Data

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
import pandas as pd, numpy as np

# Mixed data: numeric + categorical
np.random.seed(42)
df = pd.DataFrame({
    'age': np.random.randint(18, 70, 200),
    'income': np.random.normal(50000, 20000, 200),
    'city': np.random.choice(['NYC', 'LA', 'Chicago'], 200),
    'plan': np.random.choice(['basic', 'premium'], 200)
})
y = (df['income'] > 55000).astype(int)

num_cols = ['age', 'income']
cat_cols = ['city', 'plan']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first'), cat_cols)
])
pipe = Pipeline([('prep', preprocessor), ('clf', LogisticRegression())])

from sklearn.model_selection import cross_val_score
scores = cross_val_score(pipe, df, y, cv=5)
print('CV Accuracy:', scores.mean().round(4))

### Real-World Use Case

**Scenario:** HR Analytics: Build a complete pipeline to predict employee attrition from mixed numeric/categorical HR data.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pandas as pd, numpy as np

np.random.seed(42)
n = 500
df = pd.DataFrame({
    'age': np.random.randint(22, 60, n),
    'salary': np.random.normal(60000, 20000, n),
    'years_at_co': np.random.randint(1, 20, n),
    'dept': np.random.choice(['Eng', 'Sales', 'HR', 'Finance'], n),
    'satisfaction': np.random.choice(['low', 'med', 'high'], n)
})
df['attrition'] = ((df['salary'] < 45000) | (df['satisfaction'] == 'low')).astype(int)

X = df.drop('attrition', axis=1)
y = df['attrition']

pre = ColumnTransformer([
    ('num', StandardScaler(), ['age', 'salary', 'years_at_co']),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), ['dept', 'satisfaction'])
])
pipe = Pipeline([('prep', pre), ('clf', GradientBoostingClassifier(random_state=42))])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
pipe.fit(X_train, y_train)
print('Employee Attrition Pipeline:')
print(classification_report(y_test, pipe.predict(X_test)))

## Hyperparameter Tuning

Find the best model parameters using GridSearchCV and RandomizedSearchCV.

### GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=600, n_features=8, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid, cv=5, scoring='accuracy', n_jobs=-1
)
grid.fit(X_train, y_train)
print('Best params:', grid.best_params_)
print('Best CV score:', round(grid.best_score_, 4))
print('Test score:', round(grid.score(X_test, y_test), 4))

### RandomizedSearchCV (Faster)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from scipy.stats import randint, uniform

X, y = make_classification(n_samples=600, n_features=8, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

param_dist = {
    'n_estimators': randint(50, 300),
    'max_depth': randint(2, 10),
    'learning_rate': uniform(0.01, 0.3),
    'subsample': uniform(0.6, 0.4)
}

rscv = RandomizedSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_dist, n_iter=20, cv=5, scoring='accuracy',
    random_state=42, n_jobs=-1
)
rscv.fit(X_train, y_train)
print('Best params:', rscv.best_params_)
print('Best CV score:', round(rscv.best_score_, 4))
print('Test score:', round(rscv.score(X_test, y_test), 4))

### Real-World Use Case

**Scenario:** Ad Tech: Tune a click-through-rate (CTR) prediction model to maximize ROC-AUC for an ad targeting system.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from scipy.stats import randint, uniform

# Simulate CTR data (heavily imbalanced: ~2% click rate)
X, y = make_classification(
    n_samples=2000, weights=[0.98, 0.02],
    n_features=10, n_informative=6, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)

param_dist = {
    'n_estimators': randint(100, 400),
    'learning_rate': uniform(0.01, 0.2),
    'max_depth': randint(2, 6)
}
rscv = RandomizedSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_dist, n_iter=15, cv=3, scoring='roc_auc',
    random_state=42, n_jobs=-1
)
rscv.fit(X_train, y_train)
from sklearn.metrics import roc_auc_score
y_prob = rscv.predict_proba(X_test)[:, 1]
print('CTR Model Tuning:')
print('Best AUC:', round(rscv.best_score_, 4))
print('Test AUC:', round(roc_auc_score(y_test, y_prob), 4))

## Dimensionality Reduction: PCA & t-SNE

Reduce high-dimensional data for visualization, noise reduction, and speeding up training.

### PCA — Principal Component Analysis

In [ ]:
from sklearn.decomposition import PCA
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
import numpy as np

digits = load_digits()
X = StandardScaler().fit_transform(digits.data)
print('Original shape:', X.shape)  # 1797 x 64

# Keep 95% of variance
pca = PCA(n_components=0.95)
X_pca = pca.fit_transform(X)
print('Reduced shape:', X_pca.shape)
print(f'Explained variance: {pca.explained_variance_ratio_.sum():.3f}')

# 2D for visualization
pca2 = PCA(n_components=2)
X_2d = pca2.fit_transform(X)
print('2D shape:', X_2d.shape)
print('Variance explained by 2 PCs:', pca2.explained_variance_ratio_.sum().round(3))

### t-SNE for Visualization

In [ ]:
from sklearn.manifold import TSNE
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
import numpy as np

iris = load_iris()
X = StandardScaler().fit_transform(iris.data)

# t-SNE: great for visualization, non-linear
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
X_tsne = tsne.fit_transform(X)
print('t-SNE output shape:', X_tsne.shape)

# Confirm clusters align with true labels
for cls in range(3):
    mask = iris.target == cls
    center = X_tsne[mask].mean(axis=0)
    print(f'{iris.target_names[cls]}: center=({center[0]:.1f}, {center[1]:.1f})')

### Real-World Use Case

**Scenario:** NLP: Reduce TF-IDF document vectors from 5000 dims to 50 with PCA before training a classifier — 10x speedup.

In [ ]:
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# Simulated news headlines
docs = [
    'stock market rises sharply', 'fed raises interest rates',
    'tech giants report earnings', 'inflation data released',
    'new iphone model announced', 'oil prices fall today',
    'housing market cools down', 'crypto prices volatile',
    'gdp growth beats forecast', 'layoffs hit tech sector'
]
labels = [1, 1, 1, 1, 1, 0, 0, 0, 1, 0]  # 1=finance/tech, 0=other

# TF-IDF -> LSA (Truncated SVD) -> Logistic Regression
pipe = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=50)),
    ('svd', TruncatedSVD(n_components=5, random_state=42)),
    ('clf', LogisticRegression())
])
pipe.fit(docs, labels)
print('Test predictions:', pipe.predict(docs[-3:]))
print('True labels:     ', labels[-3:])